In [ ]:
# Example 1: parse_header — split a Content-Type header into main value + params dict.
import cgi

main, params = cgi.parse_header('text/html; charset=utf-8; boundary="xyz"')
print(f"main = {main!r}")
print(f"params = {params!r}")

assert main == 'text/html'
assert params['charset'] == 'utf-8'
assert params['boundary'] == 'xyz'

In [ ]:
# Example 2: cgi.parse — query-string parsing via the cgi surface.
# (cgi.parse_qs/parse_qsl were removed from stdlib cgi in Python 3.8; legacy_cgi
# tracks the final stdlib surface, so parse a GET query with cgi.parse instead.)
import cgi

qs = 'name=alice&age=30&tag=a&tag=b'
env = {'REQUEST_METHOD': 'GET', 'QUERY_STRING': qs}
parsed = cgi.parse(environ=env)
print(f"cgi.parse: {parsed}")

assert parsed['name'] == ['alice']
assert parsed['tag'] == ['a', 'b']
assert parsed['age'] == ['30']

In [ ]:
# Example 3: FieldStorage parsing a multipart/form-data payload from an in-memory stream.
import io
import cgi

boundary = '----testboundary'
body = (
    f'--{boundary}\r\n'
    'Content-Disposition: form-data; name="field1"\r\n\r\n'
    'value1\r\n'
    f'--{boundary}\r\n'
    'Content-Disposition: form-data; name="field2"\r\n\r\n'
    'value2\r\n'
    f'--{boundary}--\r\n'
).encode('utf-8')

env = {
    'REQUEST_METHOD': 'POST',
    'CONTENT_TYPE': f'multipart/form-data; boundary={boundary}',
    'CONTENT_LENGTH': str(len(body)),
}
fs = cgi.FieldStorage(fp=io.BytesIO(body), environ=env, keep_blank_values=True)
print(f"keys = {list(fs.keys())}")
print(f"field1 = {fs.getvalue('field1')!r}")
print(f"field2 = {fs.getvalue('field2')!r}")
assert fs.getvalue('field1') == 'value1'
assert fs.getvalue('field2') == 'value2'

In [ ]:
# Example 4: FieldStorage with a urlencoded POST body (the other primary content type).
import io
import cgi

body = b'username=bob&password=hunter2&remember=on'
env = {
    'REQUEST_METHOD': 'POST',
    'CONTENT_TYPE': 'application/x-www-form-urlencoded',
    'CONTENT_LENGTH': str(len(body)),
}
fs = cgi.FieldStorage(fp=io.BytesIO(body), environ=env, keep_blank_values=True)

print(f"username = {fs.getvalue('username')!r}")
print(f"password = {fs.getvalue('password')!r}")
print(f"remember = {fs.getvalue('remember')!r}")

assert fs.getvalue('username') == 'bob'
assert fs.getvalue('password') == 'hunter2'

In [ ]:
# Example 5: cgi.escape-equivalent helpers + MiniFieldStorage representation.
# cgi.escape was removed even in stdlib; legacy_cgi follows suit. Use html.escape, which
# is what the module documentation now recommends. Also exercise MiniFieldStorage.
import html
import cgi

raw = '<script>alert("xss")</script> & "quoted"'
escaped = html.escape(raw, quote=True)
print(f"escaped = {escaped!r}")
assert '&lt;script&gt;' in escaped
assert '&quot;' in escaped

mfs = cgi.MiniFieldStorage('color', 'blue')
print(f"MiniFieldStorage: name={mfs.name!r}, value={mfs.value!r}")
assert mfs.name == 'color'
assert mfs.value == 'blue'

print("\nlegacy_cgi smoke test PASSED")